# Fase II — Vehículos eléctricos: datos, estadística y probabilidad
**Estudio reproducible con datos exclusivamente simulados.** Los valores no son mediciones reales ni validan el rendimiento de ningún vehículo. El punto de partida es la Fase I: 240 kWh/día en el año 1, aumento de 12 kWh/día por año, batería de 75 kWh y 2,5 % de reducción anual sobre capacidad restante.

**Preguntas:** ¿cómo se distribuye la demanda?, ¿con qué frecuencia se supera una referencia de 280 kWh/día?, ¿qué implica el modelo porcentual de batería? El 2,5 %, la variabilidad y la tasa de incidencias son **supuestos elegidos**, no datos empíricos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
BASE = Path.cwd()
DATA = BASE / 'datos'
FIG = BASE / 'graficas'
FIG.mkdir(exist_ok=True)

## 1. Datos y organización
Hay 300 filas (60 días simulados por cada uno de 5 años). Cada fila identifica año, índice de día, temperatura supuesta (°C), demanda (kWh/día) e incidencias de carga (número de eventos por día). La segunda tabla reúne nueve valores anuales **deterministas** de salud y capacidad, de t=0 a t=8; no deben tratarse como nueve observaciones independientes reales.

Generación: demanda = 240 + 12(año−1) + 0,6(temperatura−20) + ruido normal de desviación 10 kWh/día. Temperatura uniforme de 10 a 30 °C. Incidencias Poisson(0,8), independientes. Semilla 4002. Estas decisiones son artificiales y las conclusiones solo describen el escenario simulado.

In [2]:
def cargar_datos(carpeta):
    demanda = pd.read_csv(carpeta / 'demanda_simulada.csv')
    bateria = pd.read_csv(carpeta / 'bateria_modelo.csv')
    return demanda, bateria

df, bat = cargar_datos(DATA)
print('Dimensiones:',df.shape, 'Ausentes:',int(df.isna().sum().sum()), 'Duplicados:',int(df.duplicated(['anio','dia_muestra']).sum()))
display(df.head(),bat.head())

Dimensiones: (300, 5) Ausentes: 0 Duplicados: 0


,anio,dia_muestra,temperatura_C,demanda_kWh_dia,incidencias_carga
0,1,1,21.83,243.54,0
1,1,2,14.94,243.61,0
2,1,3,26.14,231.98,2
3,1,4,18.65,223.68,0
4,1,5,16.91,247.04,1


,anio_desde_inicio,salud_pct,capacidad_kWh
0,0,100.00000,75.00000
1,1,97.50000,73.12500
2,2,95.06250,71.29688
3,3,92.68594,69.51445
4,4,90.36879,67.77659


## 2. Estadística descriptiva (P4)
La media representa el centro del consumo diario; la mediana es menos sensible a valores extremos; la desviación estándar mide la dispersión de los días. La moda no es informativa para datos casi continuos con redondeo a centésimas.

In [3]:
def resumen_por_anio(datos):
    return datos.groupby('anio')['demanda_kWh_dia'].agg(['count','mean','median','std','min','max'])

resumen = resumen_por_anio(df)
display(resumen.round(2))
print('Rango año 5:',round(resumen.loc[5,'max']-resumen.loc[5,'min'],2),'kWh/día')

,count,mean,median,std,min,max
anio,,,,,,
1,60,242.73,241.86,10.35,222.14,266.74
2,60,250.85,250.18,9.29,231.40,270.41
3,60,262.22,262.62,10.57,232.51,287.91
4,60,275.21,274.07,11.21,248.79,311.75
5,60,286.60,287.14,10.16,264.67,307.78


Rango año 5: 43.11 kWh/día


## 3. Relación entre variables
Pearson compara relaciones lineales. La fórmula de generación **impone** un vínculo entre temperatura y demanda; la correlación resultante no demuestra causalidad ni validez para vehículos reales. Al agregar años también interviene la tendencia temporal.

In [4]:
def correlacion_pearson(datos,x,y):
    r,p = stats.pearsonr(datos[x],datos[y])
    return float(r),float(p)
print('Todos los años:',correlacion_pearson(df,'temperatura_C','demanda_kWh_dia'))
print('Solo año 5:',correlacion_pearson(df.query('anio == 5'),'temperatura_C','demanda_kWh_dia'))

Todos los años: (0.06337529458562818, 0.27386284758348073)
Solo año 5: (0.2997001652354042, 0.019997908354447078)


## 4. Probabilidad (P5)
**Normal:** aproximación a demanda diaria del año 5, motivada por el ruido normal empleado al simular. **Binomial:** número de días por encima de 300 kWh/día en 20 días hipotéticos, con probabilidad constante estimada por el modelo Normal (independencia supuesta). **Poisson:** incidencias diarias, cuya tasa de 0,8 se fijó para la simulación. Ninguna representa una tasa medida.

In [5]:
def probabilidades_ingenieria(datos,umbral=300,periodo=20,lambda_inc=0.8):
    x = datos.query('anio == 5')['demanda_kWh_dia']
    mu = x.mean(); sigma = x.std(ddof=1)
    p_dia = stats.norm.sf(umbral,loc=mu,scale=sigma)
    p_diez_o_mas = stats.binom.sf(9,periodo,p_dia)
    p_dos_o_mas = stats.poisson.sf(1,lambda_inc)
    return {'media':mu,'desv':sigma,'p_dia_mas_300':p_dia,
            'p_10_o_mas_en_20':p_diez_o_mas,'p_2_o_mas_incidencias':p_dos_o_mas}
probs=probabilidades_ingenieria(df)
print({k:round(v,5) for k,v in probs.items()})

{'media': np.float64(286.60017), 'desv': np.float64(10.1567), 'p_dia_mas_300': np.float64(0.09353), 'p_10_o_mas_en_20': np.float64(0.0), 'p_2_o_mas_incidencias': np.float64(0.19121)}


## 5. Contraste de hipótesis (M2)
Pregunta: en el año 5, ¿la demanda media simulada supera 280 kWh/día? H0: μ ≤ 280; H1: μ > 280. Se emplea t de una muestra, unilateral, α=0,05, ante σ poblacional desconocida. Independencia de los 60 días simulados y residuos aproximadamente normales son hipótesis del diseño. Un valor p pequeño **no demuestra** que una flota real supere el umbral.

In [6]:
def prueba_media_superior(datos,referencia=280):
    x=datos.query('anio == 5')['demanda_kWh_dia'].to_numpy()
    test=stats.ttest_1samp(x,popmean=referencia,alternative='greater')
    inferior=x.mean()-stats.t.ppf(0.95,len(x)-1)*x.std(ddof=1)/np.sqrt(len(x))
    return float(test.statistic),float(test.pvalue),float(inferior)
t,p,limite=prueba_media_superior(df)
print('t:',round(t,3),'p unilateral:',p,'límite inferior IC 95%:',round(limite,2),'kWh/día')
print('Decisión:', 'rechazar H0' if p<0.05 else 'no rechazar H0')

t: 5.034 p unilateral: 2.410439086988006e-06 límite inferior IC 95%: 284.41 kWh/día
Decisión: rechazar H0


## 6. Figuras reproducibles
Las imágenes comparan medias anuales, modelo porcentual frente a resta lineal, distribución de demanda, correlación y días por encima de referencia. Cada figura debe ir acompañada de interpretación en el paper.

In [7]:
def guardar_grafico(nombre):
    plt.tight_layout()
    plt.savefig(FIG/nombre,dpi=180,bbox_inches='tight')
    plt.close()

plt.figure(figsize=(8,4))
plt.plot(resumen.index,resumen['mean'],marker='o',label='Promedio simulado')
plt.plot(range(1,6),[240+12*(a-1) for a in range(1,6)],'--',label='Modelo aritmético')
plt.xlabel('Año del estudio');plt.ylabel('Demanda media diaria (kWh/día)');plt.xticks(range(1,6));plt.title('Demanda energética por año');plt.legend();guardar_grafico('01_demanda_lineas.png')
plt.figure(figsize=(8,4))
plt.plot(bat['anio_desde_inicio'],bat['salud_pct'],marker='o',label='Porcentaje de capacidad restante')
plt.plot(bat['anio_desde_inicio'],100-2.5*bat['anio_desde_inicio'],'--',label='Resta lineal (no equivalente)')
plt.xlabel('Años desde inicio');plt.ylabel('Capacidad conservada (%)');plt.title('Comparación de dos hipótesis de degradación');plt.legend();guardar_grafico('02_bateria_lineas.png')
y5=df.query('anio == 5')['demanda_kWh_dia']
plt.figure(figsize=(8,4));plt.hist(y5,bins=11,edgecolor='white');plt.axvline(280,color='red',linestyle='--',label='280 kWh/día');plt.xlabel('Demanda año 5 (kWh/día)');plt.ylabel('Días simulados');plt.title('Histograma año 5');plt.legend();guardar_grafico('03_histograma_demanda.png')
plt.figure(figsize=(8,4));plt.scatter(df['temperatura_C'],df['demanda_kWh_dia'],alpha=0.6,s=16);plt.xlabel('Temperatura simulada (°C)');plt.ylabel('Demanda simulada (kWh/día)');plt.title('Dispersión de demanda y temperatura');guardar_grafico('04_dispersion_temperatura.png')
cuentas=df.assign(supera=df.demanda_kWh_dia>280).groupby('anio')['supera'].sum()
plt.figure(figsize=(8,4));plt.bar(cuentas.index.astype(str),cuentas.values);plt.xlabel('Año del estudio');plt.ylabel('Días simulados > 280 kWh/día');plt.title('Superaciones por año (60 días por año)');guardar_grafico('05_barras_superacion.png')
print('Figuras:',len(list(FIG.glob('*.png'))))

Figuras: 5


## 7. Auditoría de IA y límites
**Ejemplo de afirmación que debe verificarse:** «una correlación positiva de temperatura y demanda demuestra que el calor aumenta el consumo en todos los coches». **Auditoría:** falso; el vínculo fue impuesto en la simulación y no demuestra causalidad externa. Otro error sería afirmar que la batería durará exactamente 8,82 años: ese número proviene de una tasa constante hipotética. El estudiante debe registrar aquí la consulta real que haga a una IA, su respuesta y su propia verificación.

**Conclusión de ingeniería condicionada:** este escenario ilustra la conveniencia de revisar un umbral de abastecimiento de 280 kWh/día en el año 5. No recomienda dimensionar una instalación real sin mediciones y márgenes de seguridad.